# MDP Example 11

## Description
We want to implement the model of MDP proposed by A. Geron in the book
*Hands-On Machine Learning With Scikit-Learn and Tensorflow: Concepts, Tools, and Techniques to Build Intelligent Systems* (2017).


The MDP is given (Chapter 16 fig 16.8) by:

<img src="./geron.png" width="666">

As it can be seen, the number of actions is different in each state. We illustrate here the manner that gives matrices of same dimension for all the action.

This done by adding an action in a state:
- the transition associated to this action jumps in the same state
- the action is roughly penalized

For example the action a2 should be added is s2:
- We add a transition from s2 to s2 in the matrix associated with the action a2 (later matrix P2)
- We penalize the entry associated to state s2 action a2.

When the discount factor is 0.95

- the optimal policy is [0, 2, 1]
- the value function is [21.8992 1.17982 53.8735]

If you change the discount factor to 0.9 then

- the optimal policy should be [0, 0, 2]
- the value function should be [18.9189 0.0 50.1337]

## Tasks performed
1. Create an MDP
2. Solve the MDP
3. Check the obtained costs
4. Clean up

## Code
### Include headers

In [ ]:
#include <marmoteCore/marmoteSparseMatrix.h>
#include <marmoteCore/marmoteInterval.h>
#include <marmoteMDP/marmoteDiscountedMDP.h>
#include <marmoteMDP/marmoteFeedbackSolutionMDP.h>
#include <marmoteMDP/marmoteSolutionMDP.h>
#include <marmoteCore/marmotePolicy.h>
#include <marmoteLog/marmoteLog.h>

#include <list>
#include <vector>
#include <string>

using namespace std;


### Define parameters

In [ ]:
double beta = 0.95;
string critere("max");
double epsilon = 0.000001;
int maxIter = 700;
double penalty = -100000;

### Define the state space and the action space

In [ ]:
marmote::log::initialize();

int min = 0;
int max = 2;
int dim_SS = (max - min + 1);

MarmoteSet *actionSpace = new MarmoteInterval(0, 2);
MarmoteSet *stateSpace = new MarmoteInterval(min, max);

### Create the transition matrices
We illustrate here how we manage the creation of virtual events toward the same state.

In [ ]:
vector<TransitionStructure*> trans(actionSpace->Cardinal());

SparseMatrix *P0 = new SparseMatrix(dim_SS);
/* matrix for action a0 */
P0->setEntry(0,0,0.7);
P0->setEntry(0,1,0.3);
P0->setEntry(1,1,1.0);
P0->setEntry(2,2,1.0); /* add virtual action */
trans.at(0) = P0;

SparseMatrix *P1 = new SparseMatrix(dim_SS);
/* matrix for action a1 */
P1->setEntry(0,0,1.0);
P1->setEntry(1,2,1.0);
P1->setEntry(2,2,1.0); /* add virtual action */
trans.at(1) = P1;

SparseMatrix *P2 = new SparseMatrix(dim_SS);
/* matrix for action a2 */
P2->setEntry(0,0,0.8);
P2->setEntry(0,1,0.2);
P2->setEntry(1,1,1.0); /* add virtual action */
P2->setEntry(2,0,0.8);
P2->setEntry(2,1,0.1);
P2->setEntry(2,2,0.1);
trans.at(2) = P2;

We only illustrate here how we manage the creation of virtual events toward the same state.

### Create the reward structure

SparseMatrix object should not be filled with entries of probability 0, since this is unnecessary and less efficient.

In [ ]:
vector<TransitionStructure*> rews(actionSpace->Cardinal());

SparseMatrix *R1 = new SparseMatrix(dim_SS);
SparseMatrix *R2 = new SparseMatrix(dim_SS);
SparseMatrix *R3 = new SparseMatrix(dim_SS);

R1->setEntry(0,0,10);
R1->setEntry(2,2,penalty);

R2->setEntry(1,2,-50);
R2->setEntry(2,2,penalty);

R3->setEntry(1,1,penalty);
R3->setEntry(2,0,40);

rews.at(0) = R1;
rews.at(1) = R2;
rews.at(2) = R3;


### Build and solve the MDP

In [ ]:
DiscountedMDP *mdp1 = new DiscountedMDP(
    critere, stateSpace, actionSpace, trans, rews, beta
);

## Print the MDP

In [ ]:
mdp1->Write();

### Solve the MDP with Value Iteration

In [ ]:
FeedbackSolutionMDP *optimum = mdp1->ValueIteration(epsilon, maxIter);
optimum->Write();

### Check the obtained costs

In [ ]:
mdp1->PolicyCost(optimum, epsilon, maxIter);
for (int i = 0; i < stateSpace->Cardinal(); i++) {
    cout << "i= " << i << " sol= " << optimum->getValueIndex(i) << endl;
}

### Solve the MDP with Modified Policy Iteration

In [ ]:
SolutionMDP *optimum2 =
    mdp1->PolicyIterationModified(epsilon, maxIter, epsilon * 0.01, 100);
optimum2->Write();

### Solve the MDP with Gauss-Seidel Value Iteration

In [ ]:
SolutionMDP *optimum3 = mdp1->ValueIterationGS(epsilon, maxIter);
optimum3->Write();

### Solve the MDP with Modified Policy Iteration GS

In [ ]:
SolutionMDP *optimum4 =
    mdp1->PolicyIterationModifiedGS(epsilon, maxIter, 0.001, 20);
optimum4->Write();

In [ ]:
## Output
The notebook should produce the MDP description and the solutions returned by the different algorithms.

## Clean up

In [ ]:
delete mdp1;
delete optimum;
delete optimum2;
delete optimum3;
delete optimum4;
delete stateSpace;
delete actionSpace;

## Download
The source file is `exampleMDP11.cpp`.